In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/nyc_annualized_sales_raw.csv", low_memory=False)
print(df.shape)
df.head()

(845607, 29)


,BOROUGH,NEIGHBORHOOD,BUILDING CLASS CATEGORY,TAX CLASS AS OF FINAL ROLL,BLOCK,LOT,EASE-MENT,BUILDING CLASS AS OF FINAL ROLL,ADDRESS,APARTMENT NUMBER,...,SALE PRICE,SALE DATE,Latitude,Longitude,Community Board,Council District,BIN,BBL,Census Tract 2020,Neighborhood Tabulation Area (NTA) (2020)
0,1,CHINATOWN,47 CONDO NON-BUSINESS STORAGE,4,172,1490,NaN,RS,"50 FRANKLIN STREET, S9B",S9B,...,"$1,590,000",06/21/2019,40.716951,-74.002500,101.0,1.0,1088222.0,1.001728e+09,31.0,MN0102
1,1,EAST VILLAGE,10 COOPS - ELEVATOR APARTMENTS,2,461,24,NaN,D4,"232 EAST 6TH STREET, 6D",NaN,...,"$515,000",06/11/2019,40.727669,-73.989201,103.0,2.0,1006622.0,1.004610e+09,38.0,MN0303
2,1,FASHION,21 OFFICE BUILDINGS,4,779,23,NaN,O6,227 WEST 29TH STREET,NaN,...,$0,05/03/2019,40.748263,-73.993857,105.0,3.0,1014284.0,1.007790e+09,95.0,MN0501
3,1,FASHION,13 CONDOS - ELEVATOR APARTMENTS,2,841,1044,NaN,R4,"16 WEST 40TH STREET, 26A",26A,...,"$5,203,258",03/11/2019,40.752505,-73.982574,105.0,4.0,1090404.0,1.008418e+09,84.0,MN0502
4,1,EAST VILLAGE,10 COOPS - ELEVATOR APARTMENTS,2,431,31,NaN,D4,"141 EAST THIRD STREET, 2E",NaN,...,"$975,000",12/27/2019,40.723793,-73.985602,103.0,2.0,1005762.0,1.004310e+09,32.0,MN0303


In [2]:
pd.set_option('display.max_columns', None)
print(df.columns.tolist())
print(df.dtypes)

['BOROUGH', 'NEIGHBORHOOD', 'BUILDING CLASS CATEGORY', 'TAX CLASS AS OF FINAL ROLL', 'BLOCK', 'LOT', 'EASE-MENT', 'BUILDING CLASS AS OF FINAL ROLL', 'ADDRESS', 'APARTMENT NUMBER', 'ZIP CODE', 'RESIDENTIAL UNITS', 'COMMERCIAL UNITS', 'TOTAL UNITS', 'LAND SQUARE FEET', 'GROSS SQUARE FEET', 'YEAR BUILT', 'TAX CLASS AT TIME OF SALE', 'BUILDING CLASS AT TIME OF SALE', 'SALE PRICE', 'SALE DATE', 'Latitude', 'Longitude', 'Community Board', 'Council District', 'BIN', 'BBL', 'Census Tract 2020', 'Neighborhood Tabulation Area (NTA) (2020)']
BOROUGH                                        int64
NEIGHBORHOOD                                     str
BUILDING CLASS CATEGORY                          str
TAX CLASS AS OF FINAL ROLL                       str
BLOCK                                          int64
LOT                                            int64
EASE-MENT                                    float64
BUILDING CLASS AS OF FINAL ROLL                  str
ADDRESS                                

In [3]:
df['SALE DATE'] = pd.to_datetime(df['SALE DATE'], errors='coerce')
print(df['SALE DATE'].min(), df['SALE DATE'].max())
print(df['SALE DATE'].dt.year.value_counts().sort_index())

2016-01-01 00:00:00 2025-12-31 00:00:00
SALE DATE
2016    89806
2017    88660
2018    83782
2019    82811
2020    68684
2021    99090
2022    93427
2023    76411
2024    78243
2025    84693
Name: count, dtype: int64


Schema audit across 85 raw files

Before concatenating 17 years × 5 boroughs of DOF sales data, we check whether every file shares the same column names. Files assembled from different years/PDF sources often drift in naming — catching this now avoids silently misaligned columns after concatenation.

In [4]:
import glob

files = sorted(glob.glob("../data/raw/archive-2/**/*.csv", recursive=True))
files = [f for f in files if ".ipynb_checkpoints" not in f and "BuildingClassification" not in f]

print(f"Found {len(files)} sales files")

schema_map = {}
for f in files:
    cols = tuple(pd.read_csv(f, nrows=0).columns.tolist())
    schema_map.setdefault(cols, []).append(f)

print(f"Number of DISTINCT column schemas found: {len(schema_map)}")
for cols, flist in schema_map.items():
    print(f"\n--- {len(flist)} files with this schema ---")
    print(cols)
    print("example file:", flist[0])

Found 85 sales files
Number of DISTINCT column schemas found: 2

--- 51 files with this schema ---
('Borough', 'Neighborhood', 'Building Class Category', 'Tax Class At Present', 'Block', 'Lot', 'Ease-Ment', 'Building Class At Present', 'Address', 'Apartment Number', 'Zip Code', 'Residential Units', 'Commercial Units', 'Total Units', 'Land Square Feet', 'Gross Square Feet', 'Year Built', 'Tax Class At Time Of Sale', 'Building Class At Time Of Sale', 'Sale Price', 'Sale Date')
example file: ../data/raw/archive-2/Bronx/Bronx/03_Bronx.csv

--- 34 files with this schema ---
('Borough', 'Neighborhood', 'Building Class Category', 'Tax Class As Of Final Roll 18/1', 'Block', 'Lot', 'Ease-Ment', 'Building Class As Of Final Roll 18/1', 'Address', 'Apartment Number', 'Zip Code', 'Residential Units', 'Commercial Units', 'Total Units', 'Land Square Feet', 'Gross Square Feet', 'Year Built', 'Tax Class At Time Of Sale', 'Building Class At Time Of Sale', 'Sale Price', 'Sale Date')
example file: ../data

In [5]:
import pandas as pd

RENAME_MAP = {
    "Tax Class At Present": "Tax Class At Present",
    "Tax Class As Of Final Roll 18/1": "Tax Class At Present",
    "Building Class At Present": "Building Class At Present",
    "Building Class As Of Final Roll 18/1": "Building Class At Present",
}

dfs = []
for f in files:
    tmp = pd.read_csv(f, low_memory=False)
    tmp = tmp.rename(columns=RENAME_MAP)
    tmp["source_file"] = f  # keep provenance in case we need to trace a row back later
    dfs.append(tmp)

df_old = pd.concat(dfs, ignore_index=True)
print(df_old.shape)
print(df_old.columns.tolist())

(1626742, 22)
['Borough', 'Neighborhood', 'Building Class Category', 'Tax Class At Present', 'Block', 'Lot', 'Ease-Ment', 'Building Class At Present', 'Address', 'Apartment Number', 'Zip Code', 'Residential Units', 'Commercial Units', 'Total Units', 'Land Square Feet', 'Gross Square Feet', 'Year Built', 'Tax Class At Time Of Sale', 'Building Class At Time Of Sale', 'Sale Price', 'Sale Date', 'source_file']


In [6]:
print(df_old.dtypes)
print()
print(df_old.isnull().sum())
print()
print(df_old['Borough'].unique())
print()
df_old[['Sale Price', 'Sale Date']].head(10)

Borough                           float64
Neighborhood                          str
Building Class Category               str
Tax Class At Present                  str
Block                             float64
Lot                               float64
Ease-Ment                          object
Building Class At Present             str
Address                               str
Apartment Number                      str
Zip Code                          float64
Residential Units                 float64
Commercial Units                  float64
Total Units                       float64
Land Square Feet                  float64
Gross Square Feet                 float64
Year Built                        float64
Tax Class At Time Of Sale         float64
Building Class At Time Of Sale        str
Sale Price                        float64
Sale Date                             str
source_file                           str
dtype: object

Borough                            26580
Neighborhood        

,Sale Price,Sale Date
0,130000.0,2003-05-27 00:00:00
1,204000.0,2003-04-07 00:00:00
2,235000.0,2003-07-24 00:00:00
3,125500.0,2003-05-02 00:00:00
4,215000.0,2003-05-14 00:00:00
5,0.0,2003-07-09 00:00:00
6,245000.0,2003-04-15 00:00:00
7,500.0,2003-06-02 00:00:00
8,375000.0,2003-04-08 00:00:00
9,0.0,2003-09-10 00:00:00


In [7]:
pd.set_option('display.max_rows', None)
print(df_old.isnull().sum())

Borough                            26580
Neighborhood                       26580
Building Class Category            26580
Tax Class At Present               26824
Block                              26580
Lot                                26580
Ease-Ment                         194282
Building Class At Present          26824
Address                            26580
Apartment Number                  157516
Zip Code                           26596
Residential Units                  43806
Commercial Units                   43806
Total Units                        43806
Land Square Feet                   43807
Gross Square Feet                  43806
Year Built                         32607
Tax Class At Time Of Sale          26580
Building Class At Time Of Sale     26580
Sale Price                         26580
Sale Date                          26580
source_file                            0
dtype: int64


In [8]:
null_borough_rows = df_old[df_old['Borough'].isnull()]
print(null_borough_rows['source_file'].value_counts())
null_borough_rows.head(10)

source_file
../data/raw/archive-2/Bronx/Bronx/2018_Bronx.csv                         7015
../data/raw/archive-2/Bronx/Bronx/2019_Bronx.csv                         6594
../data/raw/archive-2/StatenIsland/StatenIsland/2019_StatenIsland.csv    6492
../data/raw/archive-2/StatenIsland/StatenIsland/2018_StatenIsland.csv    6073
../data/raw/archive-2/Brooklyn/Brooklyn/2019_Brooklyn.csv                 389
../data/raw/archive-2/Queens/Queens/2019_Queens.csv                        17
Name: count, dtype: int64


,Borough,Neighborhood,Building Class Category,Tax Class At Present,Block,Lot,Ease-Ment,Building Class At Present,Address,Apartment Number,Zip Code,Residential Units,Commercial Units,Total Units,Land Square Feet,Gross Square Feet,Year Built,Tax Class At Time Of Sale,Building Class At Time Of Sale,Sale Price,Sale Date,source_file
123058,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,../data/raw/archive-2/Bronx/Bronx/2018_Bronx.csv
123059,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,../data/raw/archive-2/Bronx/Bronx/2018_Bronx.csv
123060,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,../data/raw/archive-2/Bronx/Bronx/2018_Bronx.csv
123061,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,../data/raw/archive-2/Bronx/Bronx/2018_Bronx.csv
123062,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,../data/raw/archive-2/Bronx/Bronx/2018_Bronx.csv
123063,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,../data/raw/archive-2/Bronx/Bronx/2018_Bronx.csv
123064,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,../data/raw/archive-2/Bronx/Bronx/2018_Bronx.csv
123065,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,../data/raw/archive-2/Bronx/Bronx/2018_Bronx.csv
123066,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,../data/raw/archive-2/Bronx/Bronx/2018_Bronx.csv
123067,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,../data/raw/archive-2/Bronx/Bronx/2018_Bronx.csv


In [9]:
before = len(df_old)
df_old = df_old.dropna(subset=['Borough', 'Sale Price', 'Sale Date'], how='all')
after = len(df_old)
print(f"Dropped {before - after} fully-blank rows ({(before-after)/before:.2%} of file)")

Dropped 26580 fully-blank rows (1.63% of file)


In [10]:
print(df_old.isnull().sum())

Borough                                0
Neighborhood                           0
Building Class Category                0
Tax Class At Present                 244
Block                                  0
Lot                                    0
Ease-Ment                         167702
Building Class At Present            244
Address                                0
Apartment Number                  130936
Zip Code                              16
Residential Units                  17226
Commercial Units                   17226
Total Units                        17226
Land Square Feet                   17227
Gross Square Feet                  17226
Year Built                          6027
Tax Class At Time Of Sale              0
Building Class At Time Of Sale         0
Sale Price                             0
Sale Date                              0
source_file                            0
dtype: int64


In [11]:
mask = df_old['Tax Class At Present'].isnull()
print(df_old.loc[mask, 'source_file'].value_counts())
df_old.loc[mask].head(5)

source_file
../data/raw/archive-2/Manhattan/Manhattan/2019_Manhattan.csv             175
../data/raw/archive-2/Queens/Queens/2018_Queens.csv                       36
../data/raw/archive-2/Brooklyn/Brooklyn/2019_Brooklyn.csv                 16
../data/raw/archive-2/Brooklyn/Brooklyn/2018_Brooklyn.csv                 15
../data/raw/archive-2/Manhattan/Manhattan/2018_Manhattan.csv               1
../data/raw/archive-2/StatenIsland/StatenIsland/2018_StatenIsland.csv      1
Name: count, dtype: int64


,Borough,Neighborhood,Building Class Category,Tax Class At Present,Block,Lot,Ease-Ment,Building Class At Present,Address,Apartment Number,Zip Code,Residential Units,Commercial Units,Total Units,Land Square Feet,Gross Square Feet,Year Built,Tax Class At Time Of Sale,Building Class At Time Of Sale,Sale Price,Sale Date,source_file
544468,3.0,BEDFORD STUYVESANT,13 CONDOS - ELEVATOR APARTMENTS,NaN,1768.0,1004.0,NaN,NaN,159 TOMPKINS AVENUE,2B,11206.0,NaN,NaN,NaN,NaN,NaN,2016.0,2.0,R4,982611.0,2018-02-09 00:00:00,../data/raw/archive-2/Brooklyn/Brooklyn/2018_B...
544469,3.0,BEDFORD STUYVESANT,13 CONDOS - ELEVATOR APARTMENTS,NaN,1768.0,1005.0,NaN,NaN,159 TOMPKINS AVENUE,3A,11206.0,NaN,NaN,NaN,NaN,NaN,2016.0,2.0,R4,813581.0,2018-03-15 00:00:00,../data/raw/archive-2/Brooklyn/Brooklyn/2018_B...
544470,3.0,BEDFORD STUYVESANT,13 CONDOS - ELEVATOR APARTMENTS,NaN,1768.0,1006.0,NaN,NaN,159 TOMPKINS AVENUE,3B,11206.0,NaN,NaN,NaN,NaN,NaN,2016.0,2.0,R4,784090.0,2018-02-15 00:00:00,../data/raw/archive-2/Brooklyn/Brooklyn/2018_B...
544471,3.0,BEDFORD STUYVESANT,13 CONDOS - ELEVATOR APARTMENTS,NaN,1768.0,1007.0,NaN,NaN,159 TOMPKINS AVENUE,4A,11206.0,NaN,NaN,NaN,NaN,NaN,2016.0,2.0,R4,766935.0,2018-02-15 00:00:00,../data/raw/archive-2/Brooklyn/Brooklyn/2018_B...
544472,3.0,BEDFORD STUYVESANT,13 CONDOS - ELEVATOR APARTMENTS,NaN,1768.0,1008.0,NaN,NaN,159 TOMPKINS AVENUE,4B,11206.0,NaN,NaN,NaN,NaN,NaN,2016.0,2.0,R4,785000.0,2018-01-31 00:00:00,../data/raw/archive-2/Brooklyn/Brooklyn/2018_B...


In [12]:
print(df_old[df_old['Zip Code'].isnull()]['source_file'].value_counts())

source_file
../data/raw/archive-2/Queens/Queens/2019_Queens.csv                      7
../data/raw/archive-2/StatenIsland/StatenIsland/2019_StatenIsland.csv    6
../data/raw/archive-2/Brooklyn/Brooklyn/2018_Brooklyn.csv                2
../data/raw/archive-2/Bronx/Bronx/2019_Bronx.csv                         1
Name: count, dtype: int64


In [13]:
df_old = df_old.drop(columns=['Ease-Ment'])

In [14]:
df_old['Sale Date'] = pd.to_datetime(df_old['Sale Date'])
df_old['Borough'] = df_old['Borough'].astype(int)

print(df_old.dtypes)
print(df_old['Sale Date'].min(), df_old['Sale Date'].max())

Borough                                    int64
Neighborhood                                 str
Building Class Category                      str
Tax Class At Present                         str
Block                                    float64
Lot                                      float64
Building Class At Present                    str
Address                                      str
Apartment Number                             str
Zip Code                                 float64
Residential Units                        float64
Commercial Units                         float64
Total Units                              float64
Land Square Feet                         float64
Gross Square Feet                        float64
Year Built                               float64
Tax Class At Time Of Sale                float64
Building Class At Time Of Sale               str
Sale Price                               float64
Sale Date                         datetime64[us]
source_file         

In [15]:
COLUMN_RENAME = {
    'Borough': 'BOROUGH',
    'Neighborhood': 'NEIGHBORHOOD',
    'Building Class Category': 'BUILDING CLASS CATEGORY',
    'Tax Class At Present': 'TAX CLASS AT PRESENT',
    'Block': 'BLOCK',
    'Lot': 'LOT',
    'Building Class At Present': 'BUILDING CLASS AT PRESENT',
    'Address': 'ADDRESS',
    'Apartment Number': 'APARTMENT NUMBER',
    'Zip Code': 'ZIP CODE',
    'Residential Units': 'RESIDENTIAL UNITS',
    'Commercial Units': 'COMMERCIAL UNITS',
    'Total Units': 'TOTAL UNITS',
    'Land Square Feet': 'LAND SQUARE FEET',
    'Gross Square Feet': 'GROSS SQUARE FEET',
    'Year Built': 'YEAR BUILT',
    'Tax Class At Time Of Sale': 'TAX CLASS AT TIME OF SALE',
    'Building Class At Time Of Sale': 'BUILDING CLASS AT TIME OF SALE',
    'Sale Price': 'SALE PRICE',
    'Sale Date': 'SALE DATE',
}
df_old = df_old.rename(columns=COLUMN_RENAME)
print(sorted(df_old.columns.tolist()))

['ADDRESS', 'APARTMENT NUMBER', 'BLOCK', 'BOROUGH', 'BUILDING CLASS AT PRESENT', 'BUILDING CLASS AT TIME OF SALE', 'BUILDING CLASS CATEGORY', 'COMMERCIAL UNITS', 'GROSS SQUARE FEET', 'LAND SQUARE FEET', 'LOT', 'NEIGHBORHOOD', 'RESIDENTIAL UNITS', 'SALE DATE', 'SALE PRICE', 'TAX CLASS AT PRESENT', 'TAX CLASS AT TIME OF SALE', 'TOTAL UNITS', 'YEAR BUILT', 'ZIP CODE', 'source_file']


In [16]:
overlap_years = [2016, 2017, 2018, 2019]

old_overlap = df_old[df_old['SALE DATE'].dt.year.isin(overlap_years)]
new_overlap = df[df['SALE DATE'].dt.year.isin(overlap_years)]

print("Row counts by year — df_old (older source) vs df (newer source):")
print(pd.concat([
    old_overlap.groupby(old_overlap['SALE DATE'].dt.year).size().rename('df_old'),
    new_overlap.groupby(new_overlap['SALE DATE'].dt.year).size().rename('df')
], axis=1))

print("\nMean sale price by year (excluding $0 sales, which we'll handle properly in Phase 3):")
print(pd.concat([
    old_overlap[old_overlap['SALE PRICE'] > 0].groupby(old_overlap['SALE DATE'].dt.year)['SALE PRICE'].mean().rename('df_old'),
    new_overlap[new_overlap['SALE PRICE'] > 0].groupby(new_overlap['SALE DATE'].dt.year)['SALE PRICE'].mean().rename('df')
], axis=1))

Row counts by year — df_old (older source) vs df (newer source):
           df_old     df
SALE DATE               
2016        89806  89806
2017        88660  88660
2018        83782  83782
2019        83920  82811

Mean sale price by year (excluding $0 sales, which we'll handle properly in Phase 3):


TypeError: '>' not supported between instances of 'str' and 'int'

In [17]:
df['SALE PRICE'] = (
    df['SALE PRICE']
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)
print(df['SALE PRICE'].dtype)
print(df['SALE PRICE'].describe())

float64
count    8.456070e+05
mean     1.366774e+06
std      1.133209e+07
min      0.000000e+00
25%      0.000000e+00
50%      4.700000e+05
75%      9.390000e+05
max      2.397502e+09
Name: SALE PRICE, dtype: float64


In [18]:
new_overlap = df[df['SALE DATE'].dt.year.isin(overlap_years)]

print(pd.concat([
    old_overlap[old_overlap['SALE PRICE'] > 0].groupby(old_overlap['SALE DATE'].dt.year)['SALE PRICE'].mean().rename('df_old'),
    new_overlap[new_overlap['SALE PRICE'] > 0].groupby(new_overlap['SALE DATE'].dt.year)['SALE PRICE'].mean().rename('df')
], axis=1))

                 df_old            df
SALE DATE                            
2016       1.649424e+06  1.649424e+06
2017       1.428026e+06  1.428026e+06
2018       1.652115e+06  1.652115e+06
2019       2.035122e+06  2.042884e+06


In [19]:
# Keep only 2003-2015 from the older source; 2016-2025 comes entirely from the newer source
df_old_final = df_old[df_old['SALE DATE'].dt.year < 2016].copy()

print(f"df_old_final (2003-2015): {len(df_old_final):,} rows")
print(f"df (2016-2025): {len(df):,} rows")

combined = pd.concat([df_old_final, df], ignore_index=True, sort=False)

print(f"\nCombined total: {len(combined):,} rows")
print(f"Date range: {combined['SALE DATE'].min()} to {combined['SALE DATE'].max()}")
print(f"\nColumns: {combined.shape[1]}")
print(combined.dtypes)

df_old_final (2003-2015): 1,253,994 rows
df (2016-2025): 845,607 rows

Combined total: 2,099,601 rows
Date range: 2003-01-01 00:00:00 to 2025-12-31 00:00:00

Columns: 32
BOROUGH                                               int64
NEIGHBORHOOD                                            str
BUILDING CLASS CATEGORY                                 str
TAX CLASS AT PRESENT                                    str
BLOCK                                               float64
LOT                                                 float64
BUILDING CLASS AT PRESENT                               str
ADDRESS                                                 str
APARTMENT NUMBER                                        str
ZIP CODE                                            float64
RESIDENTIAL UNITS                                    object
COMMERCIAL UNITS                                     object
TOTAL UNITS                                          object
LAND SQUARE FEET                                  

In [20]:
pd.set_option('display.max_columns', None)
print(combined.columns.tolist())
print()
print(combined.dtypes)

['BOROUGH', 'NEIGHBORHOOD', 'BUILDING CLASS CATEGORY', 'TAX CLASS AT PRESENT', 'BLOCK', 'LOT', 'BUILDING CLASS AT PRESENT', 'ADDRESS', 'APARTMENT NUMBER', 'ZIP CODE', 'RESIDENTIAL UNITS', 'COMMERCIAL UNITS', 'TOTAL UNITS', 'LAND SQUARE FEET', 'GROSS SQUARE FEET', 'YEAR BUILT', 'TAX CLASS AT TIME OF SALE', 'BUILDING CLASS AT TIME OF SALE', 'SALE PRICE', 'SALE DATE', 'source_file', 'TAX CLASS AS OF FINAL ROLL', 'EASE-MENT', 'BUILDING CLASS AS OF FINAL ROLL', 'Latitude', 'Longitude', 'Community Board', 'Council District', 'BIN', 'BBL', 'Census Tract 2020', 'Neighborhood Tabulation Area (NTA) (2020)']

BOROUGH                                               int64
NEIGHBORHOOD                                            str
BUILDING CLASS CATEGORY                                 str
TAX CLASS AT PRESENT                                    str
BLOCK                                               float64
LOT                                                 float64
BUILDING CLASS AT PRESENT         

In [21]:
combined['TAX CLASS AT PRESENT'] = combined['TAX CLASS AT PRESENT'].fillna(combined['TAX CLASS AS OF FINAL ROLL'])
combined['BUILDING CLASS AT PRESENT'] = combined['BUILDING CLASS AT PRESENT'].fillna(combined['BUILDING CLASS AS OF FINAL ROLL'])
combined = combined.drop(columns=['TAX CLASS AS OF FINAL ROLL', 'BUILDING CLASS AS OF FINAL ROLL'])

In [22]:
for col in ['RESIDENTIAL UNITS', 'COMMERCIAL UNITS', 'TOTAL UNITS', 'LAND SQUARE FEET', 'GROSS SQUARE FEET']:
    combined[col] = (
        combined[col]
        .astype(str)
        .str.replace(',', '', regex=False)
        .replace('nan', pd.NA)
        .astype(float)
    )

print(combined[['RESIDENTIAL UNITS', 'COMMERCIAL UNITS', 'TOTAL UNITS', 'LAND SQUARE FEET', 'GROSS SQUARE FEET']].dtypes)
print(combined[['RESIDENTIAL UNITS', 'COMMERCIAL UNITS', 'TOTAL UNITS', 'LAND SQUARE FEET', 'GROSS SQUARE FEET']].describe())

RESIDENTIAL UNITS    float64
COMMERCIAL UNITS     float64
TOTAL UNITS          float64
LAND SQUARE FEET     float64
GROSS SQUARE FEET    float64
dtype: object
       RESIDENTIAL UNITS  COMMERCIAL UNITS   TOTAL UNITS  LAND SQUARE FEET  \
count       1.965935e+06      1.887854e+06  1.980239e+06      1.873548e+06   
mean        2.409479e+00      2.665646e-01  2.663402e+00      4.056395e+03   
std         3.222431e+01      7.444626e+00  3.302078e+01      3.220264e+05   
min         0.000000e+00      0.000000e+00  0.000000e+00      0.000000e+00   
25%         0.000000e+00      0.000000e+00  1.000000e+00      0.000000e+00   
50%         1.000000e+00      0.000000e+00  1.000000e+00      2.000000e+03   
75%         2.000000e+00      0.000000e+00  2.000000e+00      3.000000e+03   
max         2.081100e+04      4.964000e+03  2.081100e+04      2.147557e+08   

       GROSS SQUARE FEET  
count       1.873553e+06  
mean        4.545074e+03  
std         4.559112e+04  
min         0.000000e+00  
25%

In [23]:
combined['TAX CLASS AT PRESENT'] = combined['TAX CLASS AT PRESENT'].fillna(combined['TAX CLASS AS OF FINAL ROLL'])
combined['BUILDING CLASS AT PRESENT'] = combined['BUILDING CLASS AT PRESENT'].fillna(combined['BUILDING CLASS AS OF FINAL ROLL'])
combined = combined.drop(columns=['TAX CLASS AS OF FINAL ROLL', 'BUILDING CLASS AS OF FINAL ROLL', 'EASE-MENT'])

print(combined.shape)
print(sorted(combined.columns.tolist()))

KeyError: 'TAX CLASS AS OF FINAL ROLL'

In [24]:
pd.set_option('display.max_rows', None)
print(combined.shape)
print()
print(combined.isnull().sum())
print()
print(combined['SALE DATE'].dt.year.value_counts().sort_index())

(2099601, 30)

BOROUGH                                            0
NEIGHBORHOOD                                       0
BUILDING CLASS CATEGORY                            0
TAX CLASS AT PRESENT                            4183
BLOCK                                              0
LOT                                                0
BUILDING CLASS AT PRESENT                       4183
ADDRESS                                            0
APARTMENT NUMBER                              636936
ZIP CODE                                          67
RESIDENTIAL UNITS                             133666
COMMERCIAL UNITS                              211747
TOTAL UNITS                                   119362
LAND SQUARE FEET                              226053
GROSS SQUARE FEET                             226048
YEAR BUILT                                     39410
TAX CLASS AT TIME OF SALE                          0
BUILDING CLASS AT TIME OF SALE                     0
SALE PRICE                     

In [25]:
combined = combined.drop(columns=['EASE-MENT'], errors='ignore')
print(combined.shape)

(2099601, 29)


In [26]:
print(combined.isnull().sum())

BOROUGH                                            0
NEIGHBORHOOD                                       0
BUILDING CLASS CATEGORY                            0
TAX CLASS AT PRESENT                            4183
BLOCK                                              0
LOT                                                0
BUILDING CLASS AT PRESENT                       4183
ADDRESS                                            0
APARTMENT NUMBER                              636936
ZIP CODE                                          67
RESIDENTIAL UNITS                             133666
COMMERCIAL UNITS                              211747
TOTAL UNITS                                   119362
LAND SQUARE FEET                              226053
GROSS SQUARE FEET                             226048
YEAR BUILT                                     39410
TAX CLASS AT TIME OF SALE                          0
BUILDING CLASS AT TIME OF SALE                     0
SALE PRICE                                    

In [27]:
combined.to_csv("../data/processed/nyc_sales_2003_2025_combined.csv", index=False)
print("Saved:", combined.shape)

Saved: (2099601, 29)


In [28]:
# 1. Exact duplicate rows — could a concat or double-run have created any?
dupe_count = combined.duplicated().sum()
print(f"Exact duplicate rows: {dupe_count}")

# 2. BBL precision — still float64, needs fixing before Phase 2 uses it as an identifier
print(combined['BBL'].dropna().head())
print(combined['BBL'].dtype)

# 3. Sanity check row count math hasn't drifted from any accidental re-run
print(f"Total rows: {len(combined):,} (expected 2,099,601)")
print(f"Total columns: {combined.shape[1]} (expected 29)")

# 4. Year coverage — no gaps, no bleed past 2025
print(combined['SALE DATE'].dt.year.value_counts().sort_index())

Exact duplicate rows: 29117
1253994    1.001728e+09
1253995    1.004610e+09
1253996    1.007790e+09
1253997    1.008418e+09
1253998    1.004310e+09
Name: BBL, dtype: float64
float64
Total rows: 2,099,601 (expected 2,099,601)
Total columns: 29 (expected 29)
SALE DATE
2003    118423
2004    126586
2005    123638
2006    115467
2007    107004
2008     91142
2009     74990
2010     69031
2011     73525
2012     84081
2013     89974
2014     88687
2015     91446
2016     89806
2017     88660
2018     83782
2019     82811
2020     68684
2021     99090
2022     93427
2023     76411
2024     78243
2025     84693
Name: count, dtype: int64


In [29]:
combined['BBL'] = combined['BBL'].astype('Int64')  # nullable integer, handles the NaNs correctly
print(combined['BBL'].dropna().head())

1253994    1001727504
1253995    1004610024
1253996    1007790023
1253997    1008417501
1253998    1004310031
Name: BBL, dtype: Int64


In [30]:
combined.to_csv("../data/processed/nyc_sales_2003_2025_combined.csv", index=False)

In [31]:
dupes = combined[combined.duplicated(keep=False)].sort_values(['BOROUGH', 'BLOCK', 'LOT', 'SALE DATE'])

# Which years are duplicates concentrated in?
print(dupes['SALE DATE'].dt.year.value_counts().sort_index())

# Which source contributed them?
print(dupes['source_file'].isnull().value_counts())  # True = from the 2016-2025 file, False = from df_old

# Look at an actual duplicated pair
print(dupes.head(6).to_string())

SALE DATE
2003    2599
2004    3788
2005    3392
2006    2610
2007    2972
2008    3078
2009    3344
2010    3127
2011    2948
2012    4197
2013    3097
2014    3610
2015    3152
2016    2165
2017    1552
2018    1182
2019     160
2021     790
Name: count, dtype: int64
source_file
False    41914
True      5849
Name: count, dtype: int64
        BOROUGH               NEIGHBORHOOD                       BUILDING CLASS CATEGORY TAX CLASS AT PRESENT  BLOCK     LOT BUILDING CLASS AT PRESENT                                    ADDRESS APARTMENT NUMBER  ZIP CODE  RESIDENTIAL UNITS  COMMERCIAL UNITS  TOTAL UNITS  LAND SQUARE FEET  GROSS SQUARE FEET  YEAR BUILT  TAX CLASS AT TIME OF SALE BUILDING CLASS AT TIME OF SALE  SALE PRICE  SALE DATE                                                 source_file  Latitude  Longitude  Community Board  Council District  BIN   BBL  Census Tract 2020 Neighborhood Tabulation Area (NTA) (2020)
459491        1  UPPER BAY                  40  SELECTED GOVERNMENTAL FAC

In [32]:
before = len(combined)
combined = combined.drop_duplicates(keep='first').reset_index(drop=True)
after = len(combined)

print(f"Dropped {before - after:,} exact duplicate rows")
print(f"Final shape: {combined.shape}")

combined.to_csv("../data/processed/nyc_sales_2003_2025_combined.csv", index=False)
print("Re-saved.")

Dropped 29,117 exact duplicate rows
Final shape: (2070484, 29)
Re-saved.
